In [1]:
import ee
import geemap
ee.Authenticate()
ee.Initialize()

In [ ]:
Map = geemap.Map()

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
panama_fc = countries.filter(ee.Filter.eq("ADM0_NAME", "Panama"))
panama_geom = panama_fc.geometry()

Map.centerObject(panama_geom, 7)

In [ ]:
districts = ee.FeatureCollection("projects/deforestation-panama/assets/Districts")

In [ ]:
# Reclassified Land Cover Image
sinia_prism = ee.Image("projects/deforestation-panama/assets/landuse_2021").clip(panama_geom)

lu_index = [10,20,30,40, 50, 60, 70, 80]

VisParams = {
    "min": 10,
    "max": 80, 
    "palette": [
        '32a65e',  # [10] Mature forest
        '32a65e',  # [20] Secondary forest
        '1f8d49',  # [30] Primary forest
        '7dc975',  # [40] Crops
        '04381d',  # [50] Plantation
        '026975',  # [60] Other vegetation
        '000000',  # [70] Other landuse
        '7a6c00',  # [80] Other forest 
       ]
    }
        
    # A flat list of pixel values to replace
from_list = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33]
    # A corresponding list of new values
to_list = [10, 20, 80, 80, 80, 80, 20, 20, 60, 60, 60, 70, 70, 50, 40, 50, 40, 40, 40, 50, 40, 40, 40, 40, 50, 60, 27, 28, 29, 70, 70, 70, 70]

    # for each forest age, mask the lulc of the year immediately preceding abandonment
lulc_aggregated = ee.Image()
remapped_band = sinia_prism.remap(from_list, to_list)

# Remapped image with single band
lulc_aggregated = sinia_prism.remap(from_list, to_list).rename('landuse')

# Map pixel values to string labels for readable DataFrame outputs
class_labels = {
    '10': 'Mature forest',
    '20': 'Secondary forest',
    '30': 'Primary forest',
    '40': 'Crops',
    '50': 'Plantation',
    '60': 'Other vegetation',
    '70': 'Other landuse',
    '80': 'Other forest',
    '27': 'Water surface',
    '28': 'Populated area',
    '29': 'Infrastructure'
}